In [1]:
import pandas as pd
import numpy as np
import json 
import time 
from collections import defaultdict
from utils.train_test_split import train_test_split
from utils.neural_network import NeuralNetwork
from utils.metrics import *

# Optimal architecture
BEST_ARCHITECTURE = [30, 128, 64, 32, 1]
BEST_BATCH_SIZE = 64

# Data Split
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
RANDOM_STATE = 42

# Training configuration
MAX_EPOCHS = 200
EARLY_STOPPING_PATIENCE = 15

# Fixed threshold for evaluation
EVALUATION_THRESHOLD = 0.5

# Load the three pre-split CSV files
df_train = pd.read_csv('dataset/creditcard_train.csv')
df_val = pd.read_csv('dataset/creditcard_val.csv')
df_test = pd.read_csv('dataset/creditcard_test.csv')

# Separate features and labels
y_train = df_train['Class']
X_train = df_train.drop('Class', axis=1)

y_val = df_val['Class']
X_val = df_val.drop('Class', axis=1)

y_test = df_test['Class']
X_test = df_test.drop('Class', axis=1)

# Hyperparameter spaces for each optimizer
param_spaces = {
    'sgd': {
        'learning_rate': [0.005, 0.01, 0.05, 0.1],
        'momentum': [0.0, 0.5, 0.9, 0.99] 
    },
    'adam': {
        'learning_rate': [0.0001, 0.0005, 0.001, 0.003, 0.005, 0.01],
    },
    'adabelief': {
        'learning_rate': [0.0001, 0.0005, 0.001, 0.003, 0.005, 0.01],
    }
}

def grid_search(X_train, y_train, X_val, y_val, optimizer_name, param_space):
    """
    Perform exhaustive grid search over all hyperparameter combinations.
    Evaluate ONLY on validation set.
    
    Returns:
        best_params: dict of best hyperparameters
        search_results: list of all results
    """
    print(f"\n{'='*80}")
    print(f"Optimizer: {optimizer_name.upper()}")
    print(f"{'='*80}")
    
    # Generate all combinations of hyperparameters
    from itertools import product
    
    param_keys = list(param_space.keys())
    param_values = [param_space[key] for key in param_keys]
    
    # Create all combinations
    all_combinations = list(product(*param_values))
    total_combinations = len(all_combinations)
    
    print(f"Hyperparameter space: {dict(zip(param_keys, [len(v) for v in param_values]))}")
    print(f"Total combinations to test: {total_combinations}")
    print(f"Metric: Average Precision (PR-AUC)")
    
    # Test each parameter combination
    results = []
    
    for iter_num, combination in enumerate(all_combinations, 1):
        # Create params dict
        params = dict(zip(param_keys, combination))
        
        print(f"\n  [{iter_num}/{total_combinations}] Testing: {params}")
        
        start_time = time.time()
        
        # Train model
        nn = NeuralNetwork(
            layers=BEST_ARCHITECTURE,
            activation='relu',
            output_activation='sigmoid',
            random_state=RANDOM_STATE,
            momentum=params.get('momentum', 0.0)
        )
        
        nn.fit(
            X_train, y_train,
            X_val=X_val, y_val=y_val,
            epochs=MAX_EPOCHS,
            batch_size=BEST_BATCH_SIZE,
            learning_rate=params['learning_rate'],
            optimizer=optimizer_name,
            verbose=0,
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
        
        training_time = time.time() - start_time
        
        # Evaluate on VALIDATION SET ONLY
        y_val_pred_proba = nn.predict_proba(X_val).ravel()
        val_pr_auc = average_precision_score(y_val, y_val_pred_proba)
        val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)
        
        # Use fixed threshold for F1
        y_val_pred = (y_val_pred_proba >= EVALUATION_THRESHOLD).astype(int)
        val_f1 = f1_score(y_val, y_val_pred)
        val_precision = precision_score(y_val, y_val_pred, zero_division=0)
        val_recall = recall_score(y_val, y_val_pred, zero_division=0)
        
        print(f"    Val PR-AUC: {val_pr_auc:.4f}, Val ROC-AUC: {val_roc_auc:.4f}, Val F1: {val_f1:.4f}, Time: {training_time:.1f}s, Epochs: {len(nn.history['loss'])}")
        
        results.append({
            'params': params,
            'val_pr_auc': val_pr_auc,
            'val_roc_auc': val_roc_auc,
            'val_f1': val_f1,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'training_time': training_time,
            'epochs': len(nn.history['loss'])
        })
    
    # Find best parameters (based on PR-AUC)
    best_result = max(results, key=lambda x: x['val_pr_auc'])
    
    print(f"\n  ✓ BEST parameters: {best_result['params']}")
    print(f"    Val PR-AUC: {best_result['val_pr_auc']:.4f}")
    print(f"    Val ROC-AUC: {best_result['val_roc_auc']:.4f}")
    print(f"    Val F1: {best_result['val_f1']:.4f}")
    print(f"    Training time: {best_result['training_time']:.1f}s")
    
    return best_result['params'], results

# Run GridSearch for each optimizer
print("\nStarting grid search for all optimizers...")

# Calculate total combinations
total_combos = sum(np.prod([len(v) for v in param_spaces[opt].values()]) 
                   for opt in ['sgd', 'adam', 'adabelief'])
print(f"Total combinations across all optimizers: {total_combos}")
print(f"Expected time: ~{total_combos * 2 / 60:.0f}-{total_combos * 4 / 60:.0f} minutes")

best_params = {}
search_results = {}

for optimizer in ['sgd', 'adam', 'adabelief']:
    start_time = time.time()
    
    best_params[optimizer], search_results[optimizer] = grid_search(
        X_train,
        y_train,
        X_val,
        y_val,
        optimizer,
        param_spaces[optimizer]
    )
    
    elapsed = time.time() - start_time
    print(f"\n  Total time for {optimizer.upper()}: {elapsed/60:.1f} minutes")

# Save search results
search_results_save = {
    opt: [{'params': str(r['params']), 
           'val_pr_auc': float(r['val_pr_auc']), 
           'val_roc_auc': float(r['val_roc_auc']), 
           'val_f1': float(r['val_f1'])} for r in results]
    for opt, results in search_results.items()
}

with open('hyperparameter_search_results_robust.json', 'w') as f:
    json.dump(search_results_save, f, indent=2)

with open('best_hyperparameters.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print(f"\n✓ Search results saved to: hyperparameter_search_results.json")
print(f"✓ Best hyperparameters saved to: best_hyperparameters.json")

# Retrain with Best Hyperparameters
trained_models = {}
validation_results = {}

for optimizer in ['sgd', 'adam', 'adabelief']:
    start_time = time.time()
    
    # Train model
    nn = NeuralNetwork(
        layers=BEST_ARCHITECTURE,
        activation='relu',
        output_activation='sigmoid',
        random_state=RANDOM_STATE,
        momentum=best_params.get('momentum', 0.0)
    )
    
    fit_kwargs = dict(
        X_val=X_val, y_val=y_val,
        epochs=MAX_EPOCHS,
        batch_size=BEST_BATCH_SIZE,
        learning_rate=best_params[optimizer]['learning_rate'],
        optimizer=optimizer,
        verbose=0,
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )

    nn.fit(X_train, y_train, **fit_kwargs)
    
    training_time = time.time() - start_time
    epochs_trained = len(nn.history['loss'])

    # Evaluate on Validation Set Only
    y_val_pred_proba = nn.predict_proba(X_val).ravel()
    y_val_pred = (y_val_pred_proba >= EVALUATION_THRESHOLD).astype(int)

    # Calculate metrics
    cm = confusion_matrix(y_val, y_val_pred)
    tn, fp, fn, tp = cm.ravel()

    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred, zero_division=0)
    val_recall = recall_score(y_val, y_val_pred, zero_division=0)
    val_f1 = f1_score(y_val, y_val_pred, zero_division=0)
    val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)
    val_pr_auc = average_precision_score(y_val, y_val_pred_proba)

    # Store results
    trained_models[optimizer] = nn
    validation_results[optimizer] = {
        'best_params': best_params[optimizer],
        'training_time': training_time,
        'epochs_trained': epochs_trained,
        'validation_metrics': {
            'pr_auc': val_pr_auc,
            'roc_auc': val_roc_auc,
            'f1_score': val_f1,
            'precision': val_precision,
            'recall': val_recall,
            'accuracy': val_accuracy,
            'confusion_matrix': {
                'tn': int(tn), 'fp': int(fp),
                'fn': int(fn), 'tp': int(tp)
            }
        }
    }

    print(f"  Val PR-AUC:    {val_pr_auc:.4f}")
    print(f"  Val ROC-AUC:   {val_roc_auc:.4f}")
    print(f"  Val F1-Score:  {val_f1:.4f}")
    print(f"  Val Precision: {val_precision:.4f}")
    print(f"  Val Recall:    {val_recall:.4f}")
    print(f"  Val Accuracy:  {val_accuracy:.4f}")
    print(f"\n  Confusion Matrix (Validation):")
    print(f"    TP: {tp:,}  FP: {fp:,}")
    print(f"    FN: {fn:,}  TN: {tn:,}")

# Save validation results
with open('optimizer_comparison_validation_results_robust.json', 'w') as f:
    json.dump(validation_results, f, indent=2, default=float)

print(f"\n✓ Validation results saved to: optimizer_comparison_validation_results.json")

# Create comparison table
metrics_table = []
# SESUDAH (fixed)
for opt in ['sgd', 'adam', 'adabelief']:
    r = validation_results[opt]['validation_metrics']
    params = validation_results[opt]['best_params']
    
    metrics_table.append({
        'Optimizer': opt.upper(),
        'Learning Rate': params['learning_rate'],
        'Momentum': params.get('momentum', '-'),  # '-' jika tidak ada (Adam/AdaBelief)
        'Val PR-AUC': f"{r['pr_auc']:.4f}",
        'Val ROC-AUC': f"{r['roc_auc']:.4f}",
        'Val F1-Score': f"{r['f1_score']:.4f}",
        'Val Precision': f"{r['precision']:.4f}",
        'Val Recall': f"{r['recall']:.4f}",
        'Training Time': f"{validation_results[opt]['training_time']:.1f}s",
        'Epochs': validation_results[opt]['epochs_trained']
    })

df_comparison = pd.DataFrame(metrics_table)

print(f"\n{df_comparison.to_string(index=False)}")

# Save comparison table
df_comparison.to_csv('optimizer_comparison_validation_table_robust.csv', index=False)
print(f"\n✓ Comparison table saved to: optimizer_comparison_validation_table.csv")

# Find best optimizer based on Val PR-AUC
best_optimizer = max(validation_results.keys(), 
                     key=lambda x: validation_results[x]['validation_metrics']['pr_auc'])



Starting grid search for all optimizers...
Total combinations across all optimizers: 28
Expected time: ~1-2 minutes

Optimizer: SGD
Hyperparameter space: {'learning_rate': 4, 'momentum': 4}
Total combinations to test: 16
Metric: Average Precision (PR-AUC)

  [1/16] Testing: {'learning_rate': 0.005, 'momentum': 0.0}
    Val PR-AUC: 0.8563, Val ROC-AUC: 0.9903, Val F1: 0.8507, Time: 79.9s, Epochs: 136

  [2/16] Testing: {'learning_rate': 0.005, 'momentum': 0.5}
    Val PR-AUC: 0.8596, Val ROC-AUC: 0.9908, Val F1: 0.8551, Time: 50.7s, Epochs: 80

  [3/16] Testing: {'learning_rate': 0.005, 'momentum': 0.9}
    Val PR-AUC: 0.8580, Val ROC-AUC: 0.9874, Val F1: 0.8529, Time: 23.4s, Epochs: 37

  [4/16] Testing: {'learning_rate': 0.005, 'momentum': 0.99}
    Val PR-AUC: 0.8270, Val ROC-AUC: 0.9823, Val F1: 0.8194, Time: 11.6s, Epochs: 19

  [5/16] Testing: {'learning_rate': 0.01, 'momentum': 0.0}
    Val PR-AUC: 0.8599, Val ROC-AUC: 0.9908, Val F1: 0.8551, Time: 48.2s, Epochs: 80

  [6/16] Te

In [2]:
import pandas as pd
import os

os.makedirs('results', exist_ok=True)

# ============================================================
# 1. SIMPAN VALIDATION RESULTS
# ============================================================

val_rows = []
for opt in ['sgd', 'adam', 'adabelief']:
    r = validation_results[opt]['validation_metrics']
    params = validation_results[opt]['best_params']
    cm = r['confusion_matrix']
    
    val_rows.append({
        'Optimizer': opt.upper(),
        'Learning Rate': params['learning_rate'],
        'Momentum': params.get('momentum', '-'),
        'Val PR-AUC': round(r['pr_auc'], 4),
        'Val ROC-AUC': round(r['roc_auc'], 4),
        'Val F1-Score': round(r['f1_score'], 4),
        'Val Precision': round(r['precision'], 4),
        'Val Recall': round(r['recall'], 4),
        'Val Accuracy': round(r['accuracy'], 4),
        'TP': cm['tp'], 'FP': cm['fp'],
        'FN': cm['fn'], 'TN': cm['tn'],
        'Training Time (s)': round(validation_results[opt]['training_time'], 1),
        'Epochs': validation_results[opt]['epochs_trained']
    })

df_val_results = pd.DataFrame(val_rows)
df_val_results.to_csv('results/validation_results.csv', index=False)
print("✓ Saved: results/validation_results.csv")

# ============================================================
# 2. TEST SET EVALUATION
# ============================================================

y_test = df_test['Class']
X_test = df_test.drop('Class', axis=1)

test_results = {}
test_rows = []

for opt in ['sgd', 'adam', 'adabelief']:
    nn = trained_models[opt]
    params = validation_results[opt]['best_params']

    # Predict on test set
    y_test_pred_proba = nn.predict_proba(X_test).ravel()
    y_test_pred = (y_test_pred_proba >= EVALUATION_THRESHOLD).astype(int)

    # Metrics
    cm = confusion_matrix(y_test, y_test_pred)
    tn, fp, fn, tp = cm.ravel()

    test_pr_auc   = average_precision_score(y_test, y_test_pred_proba)
    test_roc_auc  = roc_auc_score(y_test, y_test_pred_proba)
    test_f1       = f1_score(y_test, y_test_pred, zero_division=0)
    test_precision = precision_score(y_test, y_test_pred, zero_division=0)
    test_recall   = recall_score(y_test, y_test_pred, zero_division=0)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    test_results[opt] = {
        'pr_auc': test_pr_auc, 'roc_auc': test_roc_auc,
        'f1_score': test_f1, 'precision': test_precision,
        'recall': test_recall, 'accuracy': test_accuracy,
        'confusion_matrix': {'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)}
    }

    test_rows.append({
        'Optimizer': opt.upper(),
        'Learning Rate': params['learning_rate'],
        'Momentum': params.get('momentum', '-'),
        'Test PR-AUC': round(test_pr_auc, 4),
        'Test ROC-AUC': round(test_roc_auc, 4),
        'Test F1-Score': round(test_f1, 4),
        'Test Precision': round(test_precision, 4),
        'Test Recall': round(test_recall, 4),
        'Test Accuracy': round(test_accuracy, 4),
        'TP': int(tp), 'FP': int(fp),
        'FN': int(fn), 'TN': int(tn)
    })

    print(f"\n{'='*40}")
    print(f"Optimizer: {opt.upper()}")
    print(f"  Test PR-AUC:    {test_pr_auc:.4f}")
    print(f"  Test ROC-AUC:   {test_roc_auc:.4f}")
    print(f"  Test F1-Score:  {test_f1:.4f}")
    print(f"  Test Precision: {test_precision:.4f}")
    print(f"  Test Recall:    {test_recall:.4f}")
    print(f"  Test Accuracy:  {test_accuracy:.4f}")
    print(f"  Confusion Matrix:")
    print(f"    TP: {tp:,}  FP: {fp:,}")
    print(f"    FN: {fn:,}  TN: {tn:,}")

df_test_results = pd.DataFrame(test_rows)
df_test_results.to_csv('results/test_results.csv', index=False)
print("\n✓ Saved: results/test_results.csv")

# ============================================================
# 3. SIMPAN COMBINED COMPARISON (VAL + TEST)
# ============================================================

combined_rows = []
for opt in ['sgd', 'adam', 'adabelief']:
    v = validation_results[opt]['validation_metrics']
    t = test_results[opt]
    params = validation_results[opt]['best_params']

    combined_rows.append({
        'Optimizer': opt.upper(),
        'Learning Rate': params['learning_rate'],
        'Momentum': params.get('momentum', '-'),
        'Val PR-AUC': round(v['pr_auc'], 4),
        'Val ROC-AUC': round(v['roc_auc'], 4),
        'Val F1-Score': round(v['f1_score'], 4),
        'Val Precision': round(v['precision'], 4),
        'Val Recall': round(v['recall'], 4),
        'Test PR-AUC': round(t['pr_auc'], 4),
        'Test ROC-AUC': round(t['roc_auc'], 4),
        'Test F1-Score': round(t['f1_score'], 4),
        'Test Precision': round(t['precision'], 4),
        'Test Recall': round(t['recall'], 4),
    })

df_combined = pd.DataFrame(combined_rows)
df_combined.to_csv('results/combined_val_test_results.csv', index=False)
print("✓ Saved: results/combined_val_test_results.csv")

✓ Saved: results/validation_results.csv

Optimizer: SGD
  Test PR-AUC:    0.8086
  Test ROC-AUC:   0.9558
  Test F1-Score:  0.8154
  Test Precision: 0.8833
  Test Recall:    0.7571
  Test Accuracy:  0.9994
  Confusion Matrix:
    TP: 53  FP: 7
    FN: 17  TN: 42,480

Optimizer: ADAM
  Test PR-AUC:    0.7681
  Test ROC-AUC:   0.9689
  Test F1-Score:  0.7857
  Test Precision: 0.7857
  Test Recall:    0.7857
  Test Accuracy:  0.9993
  Confusion Matrix:
    TP: 55  FP: 15
    FN: 15  TN: 42,472

Optimizer: ADABELIEF
  Test PR-AUC:    0.8083
  Test ROC-AUC:   0.9624
  Test F1-Score:  0.8029
  Test Precision: 0.8209
  Test Recall:    0.7857
  Test Accuracy:  0.9994
  Confusion Matrix:
    TP: 55  FP: 12
    FN: 15  TN: 42,475

✓ Saved: results/test_results.csv
✓ Saved: results/combined_val_test_results.csv


In [3]:
# ============================================================
# SIMPAN GRID SEARCH RESULTS
# ============================================================

os.makedirs('results', exist_ok=True)

# Simpan per optimizer
for optimizer in ['sgd', 'adam', 'adabelief']:
    rows = []
    for r in search_results[optimizer]:
        row = {
            'Optimizer': optimizer.upper(),
            'Learning Rate': r['params']['learning_rate'],
            'Momentum': r['params'].get('momentum', '-'),
            'Val PR-AUC': round(r['val_pr_auc'], 4),
            'Val ROC-AUC': round(r['val_roc_auc'], 4),
            'Val F1-Score': round(r['val_f1'], 4),
            'Val Precision': round(r['val_precision'], 4),
            'Val Recall': round(r['val_recall'], 4),
            'Training Time (s)': round(r['training_time'], 1),
            'Epochs': r['epochs'],
            'Is Best': r['params'] == best_params[optimizer]
        }
        rows.append(row)

    df = pd.DataFrame(rows).sort_values('Val PR-AUC', ascending=False)
    df.to_csv(f'results/grid_search_{optimizer}.csv', index=False)
    print(f"✓ Saved: results/grid_search_{optimizer}.csv")

# Simpan semua optimizer dalam satu file
all_rows = []
for optimizer in ['sgd', 'adam', 'adabelief']:
    for r in search_results[optimizer]:
        all_rows.append({
            'Optimizer': optimizer.upper(),
            'Learning Rate': r['params']['learning_rate'],
            'Momentum': r['params'].get('momentum', '-'),
            'Val PR-AUC': round(r['val_pr_auc'], 4),
            'Val ROC-AUC': round(r['val_roc_auc'], 4),
            'Val F1-Score': round(r['val_f1'], 4),
            'Val Precision': round(r['val_precision'], 4),
            'Val Recall': round(r['val_recall'], 4),
            'Training Time (s)': round(r['training_time'], 1),
            'Epochs': r['epochs'],
            'Is Best': r['params'] == best_params[optimizer]
        })

df_all = pd.DataFrame(all_rows)
df_all.to_csv('results/grid_search_all.csv', index=False)
print("✓ Saved: results/grid_search_all.csv")

# Print ringkasan best params per optimizer
print("\n=== BEST HYPERPARAMETERS ===")
for optimizer in ['sgd', 'adam', 'adabelief']:
    best = df_all[(df_all['Optimizer'] == optimizer.upper()) & (df_all['Is Best'] == True)]
    print(f"\n{optimizer.upper()}: {best[['Learning Rate', 'Momentum', 'Val PR-AUC']].to_string(index=False)}")

✓ Saved: results/grid_search_sgd.csv
✓ Saved: results/grid_search_adam.csv
✓ Saved: results/grid_search_adabelief.csv
✓ Saved: results/grid_search_all.csv

=== BEST HYPERPARAMETERS ===

SGD:  Learning Rate Momentum  Val PR-AUC
          0.01      0.0      0.8599

ADAM:  Learning Rate Momentum  Val PR-AUC
        0.0005        -       0.868

ADABELIEF:  Learning Rate Momentum  Val PR-AUC
        0.0001        -      0.8433
